# Akita — Predicting 3D Genome Folding from DNA Sequence · Methodology

This notebook documents the **methodology** used to run inference with **Akita**, a deep
convolutional neural network that predicts 3D genome folding (Hi-C / Micro-C contact maps)
directly from DNA sequence.

**Paper:** Fudenberg, Kelley & Pollard, *Predicting 3D genome folding from DNA sequence with Akita*, **Nature Methods** 17, 1111–1117 (2020).
**Code:** [calico/basenji → manuscripts/akita](https://github.com/calico/basenji/tree/master/manuscripts/akita)

Akita takes a **1 Mb (2²⁰ bp) DNA window** as input and predicts the **contact map**
(log observed/expected) for that locus — the visual "genome folding" result from the paper —
across several cell types, using the authors' **pre-trained** weights (**inference only, no training**).

The runnable pipeline lives in [`akita_inference_colab.ipynb`](./akita_inference_colab.ipynb);
this notebook explains *why* each step is done and how the pieces fit together.

## 1. Overview of the approach

The genome is not a linear string — it folds in 3D so that distant regulatory elements
(enhancers, promoters, CTCF insulators) come into contact. Experimental **Hi-C** and **Micro-C**
assays measure these contacts as a symmetric matrix. Akita learns the mapping

$$\text{DNA sequence (1 Mb)} \;\longrightarrow\; \text{contact map}$$

directly, so a folding prediction can be made for **any** locus — including sequences that were
never assayed, or sequences carrying engineered variants.

**Methodology at a glance:**

1. Encode a 2²⁰ bp genomic window as a one-hot matrix.
2. Pass it through the pre-trained convolutional network.
3. Reshape the flattened prediction back into a symmetric contact matrix.
4. Visualise and interpret the predicted folding (TADs, boundaries, loops).

## 2. Environment setup and how to run

The pipeline is **inference only** and runs on **CPU** — a **Google Colab** CPU runtime (or a local
Linux/WSL2 environment) is enough; no GPU is needed for a single locus. The steps below mirror the
runnable [`akita_inference_colab.ipynb`](./akita_inference_colab.ipynb); run them top to bottom.

> **Dependency note.** basenji pins older library versions, so the install may surface version
> warnings. If the imports in the last cell fail right after installing: **Runtime > Restart session**,
> then re-run from the *Imports* cell (skip the install cell). The first run also needs two one-time
> downloads — the trained weights (~a few MB) and the hg38 FASTA (~1 GB).

In [ ]:
# Step 1 — install Akita (basenji) + dependencies. Takes a few minutes on Colab.
!git clone --depth 1 https://github.com/calico/basenji.git /content/basenji
!pip install -q /content/basenji
!pip install -q cooltools pysam

In [ ]:
# Step 2 — move into the akita manuscript dir (params.json + data/ live here)
#           and download the pre-trained weights (human Hi-C + Micro-C, hg38, 2048 bp).
import os
os.chdir('/content/basenji/manuscripts/akita')
if not os.path.isfile('model_best.h5'):
    !wget -q https://storage.googleapis.com/basenji_hic/1m/models/9-14/model_best.h5
print('model present:', os.path.isfile('model_best.h5'))

In [ ]:
# Step 3 — get the hg38 sequence source (~1 GB, one-time download).
import pysam
if not os.path.isfile('data/hg38.ml.fa'):
    !curl -o data/hg38.ml.fa.gz https://storage.googleapis.com/basenji_barnyard2/hg38.ml.fa.gz
    !gunzip data/hg38.ml.fa.gz
fasta_open = pysam.Fastafile('data/hg38.ml.fa')
print('fasta ready')

In [ ]:
# Step 4 — imports. Force CPU first (robust; avoids GPU/cuDNN issues), then load the libraries.
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
os.environ["CUDA_VISIBLE_DEVICES"] = '-1'
import tensorflow as tf
from cooltools.lib.numutils import set_diag
from basenji import dna_io, seqnn

## 3. Input representation — one-hot DNA

The model consumes a fixed **2²⁰ = 1,048,576 bp** window (~1 Mb). Each base is one-hot encoded
into 4 channels (A, C, G, T), giving an array of shape **[1,048,576, 4]**. `N`/unknown bases
map to an all-zero column.

The window length is *fixed*: the locus must span exactly 2²⁰ bp or the model will not accept it.

In [ ]:
# From akita_inference_colab.ipynb — encode the chosen locus
region = 'chr12:115163136-116211712'   # example locus from the paper (exactly 2^20 bp wide)

chrm, coords = region.split(':')
seq_start, seq_end = (int(x) for x in coords.split('-'))
assert seq_end - seq_start == seq_length, f'region must span exactly {seq_length} bp'

seq = fasta_open.fetch(chrm, seq_start, seq_end).upper()
seq_1hot = dna_io.dna_1hot(seq)          # shape [1048576, 4]

## 4. Model architecture

Akita is built on the **Basenji** convolutional framework and has two conceptual stages:

**(a) 1D convolutional trunk.**
A stack of convolution + pooling layers reads the 1 Mb sequence and progressively **downsamples**
it from single-base resolution to **2,048 bp bins** (`pool_width = 2048`), while widening the
receptive field with **dilated convolutions** so each bin summarises a long stretch of context.
The trunk outputs a 1D feature vector along the binned sequence (≈512 bins).

**(b) 2D head.**
The 1D features are turned into a 2D representation by combining every pair of bins *(i, j)*
(an outer/broadcast operation), then refined with **dilated 2D convolutions** and **symmetrised**
(a contact map must satisfy *M(i, j) = M(j, i)*). The head outputs one predicted map per target
cell type.

**Target / output format.**
Because the map is symmetric, only the **upper triangle** is predicted, stored as a flat vector,
with the first few diagonals (`diagonal_offset`) masked (the strong self-contact diagonal is not
modelled). The edges are **cropped** (`crop_bp`) because bins near the window boundary lack full
sequence context. The prediction target is the **log(observed / expected)** contact value at
**2,048 bp resolution**.

In [ ]:
# Load the pre-trained model (from akita_inference_colab.ipynb)
with open('params.json') as f:
    params = json.load(f)
seqnn_model = seqnn.SeqNN(params['model'])
seqnn_model.restore('model_best.h5')   # human Hi-C + Micro-C weights, hg38, 2048 bp resolution

## 5. Reshape constants and target cell types

The dataset statistics (`data/statistics.json`) define how the flat prediction maps back onto a
square matrix, and `data/targets.txt` lists the experimental **cell types / assays** the model
predicts simultaneously (multitask output).

In [ ]:
# Derive the reshape geometry (from akita_inference_colab.ipynb)
hic_targets = pd.read_csv('data/targets.txt', sep='\t')
hic_num_to_name = dict(zip(hic_targets['index'].values, hic_targets['identifier'].values))

with open('data/statistics.json') as f:
    data_stats = json.load(f)
seq_length     = data_stats['seq_length']                       # 2^20 = 1,048,576 bp input
hic_diags      = data_stats['diagonal_offset']                  # masked central diagonals
target_crop    = data_stats['crop_bp'] // data_stats['pool_width']
target_length1 = data_stats['seq_length'] // data_stats['pool_width']   # ~512 bins
target_length1_cropped = target_length1 - 2 * target_crop       # size of the final square map

## 6. From flat prediction back to a symmetric matrix

The network outputs the upper triangle as a 1D vector. The helper below scatters that vector into
the upper triangle of a square array, masks the central diagonals with `NaN`, and mirrors it
(`z + z.T`) to rebuild the full **symmetric contact map**.

In [ ]:
def from_upper_triu(vector_repr, matrix_len, num_diags):
    z = np.zeros((matrix_len, matrix_len))
    triu_tup = np.triu_indices(matrix_len, num_diags)
    z[triu_tup] = vector_repr
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)
    return z + z.T

## 7. Inference

The model expects a **batch**: the one-hot window is expanded to shape `[1, 2²⁰, 4]`.
The output has shape `[1, #pixels, #cell types]`, where `#pixels` is the length of the
upper-triangle vector.

In [ ]:
pred = seqnn_model.model.predict(np.expand_dims(seq_1hot, 0))   # [1, #pixels, #cell types]

## 8. Visualising the predicted contact map

Pick a target cell type, rebuild the square matrix, and plot it. A **diverging RdBu** colormap
centred at 0 is used because the values are **log(observed/expected)**: **red = more contact than
expected**, **blue = less**. The colour scale is clamped to `[-2, 2]`.

In [ ]:
target_index = 0        # 0 = first cell type in data/targets.txt
vmin, vmax = -2, 2

mat = from_upper_triu(pred[:, :, target_index], target_length1_cropped, hic_diags)

plt.figure(figsize=(6, 6))
im = plt.matshow(mat, fignum=False, cmap='RdBu_r', vmin=vmin, vmax=vmax)
plt.colorbar(im, fraction=.04, pad=0.05, ticks=[-2, -1, 0, 1, 2])
plt.title(f'Akita prediction - {hic_num_to_name[target_index]}\n{region}', y=1.08)
plt.savefig('akita_pred.png', dpi=150, bbox_inches='tight')
plt.show()

### How to read the output

- The map is **symmetric** about the diagonal; the diagonal itself is masked (self-contact).
- **Blocks of red along the diagonal** are **Topologically Associating Domains (TADs)** — regions
  that self-interact.
- Sharp changes between blocks are **domain boundaries**, typically anchored by **CTCF** sites.
- Off-diagonal **red dots** are **loops** — specific long-range contacts (often CTCF–CTCF).

The predicted map recreates these features from sequence alone, matching the qualitative structure
reported in the paper.

## 9. Why this matters for variant prioritisation

Because Akita predicts folding **from sequence**, it doubles as an **in-silico mutagenesis** tool
for our team's variant-prioritisation goal:

1. Predict the contact map for the **reference** sequence.
2. Introduce a candidate **variant** (SNV, indel, inversion, or a CTCF-motif orientation flip).
3. Predict the contact map for the **altered** sequence.
4. Score the variant by the **difference** between the two maps.

Variants that reshape TAD boundaries or abolish loops (e.g. disrupting a CTCF site) produce a large
predicted change and can be **ranked as high-priority** structural/regulatory variants — even when
they fall in non-coding DNA that sequence-only variant callers would overlook. A minimal, runnable
illustration of this reference-vs-variant comparison (flipping a CTCF motif from convergent to
divergent) is shown in [`akita_contactmap_demo.ipynb`](./akita_contactmap_demo.ipynb).

## 10. Summary of the methodology

| Step | What happens | Key object |
|------|--------------|-----------|
| 0 | Install basenji + download weights & hg38 FASTA (CPU/Colab) | `model_best.h5`, `hg38.ml.fa` |
| 1 | Select a 2²⁰ bp hg38 window | `region` |
| 2 | One-hot encode the DNA | `seq_1hot` — `[1048576, 4]` |
| 3 | Load pre-trained Akita weights | `model_best.h5` |
| 4 | Forward pass (inference) | `pred` — `[1, #pixels, #cell types]` |
| 5 | Rebuild symmetric matrix | `from_upper_triu(...)` |
| 6 | Plot log(obs/exp) contact map | `akita_pred.png` |

**Inference only** with the authors' pre-trained model; CPU is sufficient for a single locus.
The pipeline turns raw DNA into an interpretable 3D-folding prediction and provides a principled way
to score how sequence variants perturb genome architecture.